# Cell 0 - Python Structured Specialist Dataset Factory

## Qwen Teacher + Open-Source Python Data + New-Library Generalization

This notebook builds training data for the planned **Python Structured Specialist**.

- **Student/base model:** `Salesforce/codet5p-220m-py`
- **Teacher:** `Qwen/Qwen2.5-Coder-7B-Instruct` in 4-bit on a Colab T4
- **Student target:** formal **Python-oriented IR**
- **Final user-visible output later:** normal Python source

The final dataset has 24 columns:

```text
id
task
instruction
code_before
error
project_context
library_context
tests
target_mode
target_ir
target_code
domain
subdomain
difficulty
instruction_style
concepts
libraries
verified
verification_level
source_type
source_dataset
source_reference
ir_version
content_hash
```

Three pipelines are included:

1. Qwen teacher synthesis for Generate, Full-file Generate, Repair, Refactor.
2. Public Python data transformation from Magicoder OSS-Instruct and CommitPackFT.
3. New/unfamiliar library synthesis using fictional package APIs supplied through `library_context`.

The notebook is resumable and exports JSONL, CSV, Parquet, and a generation manifest.

> Colab: Runtime -> Change runtime type -> T4 GPU.

# Cell 1 - Install dependencies

The teacher is loaded with 4-bit NF4 quantization. Generation does **not** pass `max_new_tokens`; it uses a total-sequence safety ceiling and allows EOS to stop naturally.

In [ ]:
!pip install -q -U \
    "transformers>=4.46" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44" \
    "datasets>=3.0" \
    "huggingface_hub>=0.26" \
    sentencepiece safetensors pandas pyarrow tqdm pytest pytest-asyncio

# Cell 2 - Imports and reproducibility

In [ ]:
import ast
import csv
import gc
import hashlib
import json
import os
import random
import re
import sqlite3
import subprocess
import sys
import tempfile
import textwrap
import time
from collections import Counter
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

# Cell 3 - Runtime configuration

`DEMO_MODE=True` keeps the first run small. After inspecting quality, change it to `False` and increase targets gradually.

Mature dataset target:

| Task | Rows |
|---|---:|
| Generate | 50,000 |
| Full-file Generate | 50,000 |
| Repair | 50,000 |
| Refactor | 50,000 |
| **Total** | **200,000** |

In [ ]:
STUDENT_MODEL_ID = "Salesforce/codet5p-220m-py"
TEACHER_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

DEMO_MODE = True
GENERATION_MAX_TOTAL_TOKENS = 8192
TEACHER_RETRIES_PER_ROW = 3
RUN_GENERATED_TESTS = False
PYTEST_TIMEOUT_SECONDS = 20
STRICT_STRUCTURE_DEDUP = False

NEW_LIBRARY_MIN_FRACTION = 0.10
NEW_LIBRARY_IDEAL_FRACTION = 0.15
NEW_LIBRARY_MAX_FRACTION = 0.20

if DEMO_MODE:
    QWEN_TARGET_PER_TASK = 2
    MAGICODER_ACCEPTED_TARGET = 8
    COMMITPACK_ACCEPTED_TARGET = 8
    NEW_LIBRARY_ACCEPTED_TARGET = 8
else:
    QWEN_TARGET_PER_TASK = 500
    MAGICODER_ACCEPTED_TARGET = 5000
    COMMITPACK_ACCEPTED_TARGET = 5000
    NEW_LIBRARY_ACCEPTED_TARGET = 2000

FINAL_TASK_TARGETS = {
    "generate": 50000,
    "full_file_generate": 50000,
    "repair": 50000,
    "refactor": 50000,
}
TASKS = tuple(FINAL_TASK_TARGETS)
IR_VERSION = "v1"
MIN_CODE_CHARS = 30
MAX_CODE_CHARS = 40000

print("Student:", STUDENT_MODEL_ID)
print("Teacher:", TEACHER_MODEL_ID)
print("Demo mode:", DEMO_MODE)

# Cell 4 - Storage and optional Google Drive

For long Colab runs, set `MOUNT_DRIVE=True` so checkpoints survive runtime resets.

In [ ]:
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT_DIR = Path('/content/drive/MyDrive/python_structured_specialist_dataset')
else:
    ROOT_DIR = Path('/content/python_structured_specialist_dataset')

ROOT_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_JSONL = ROOT_DIR / 'teacher_qwen.jsonl'
MAGICODER_JSONL = ROOT_DIR / 'public_magicoder.jsonl'
COMMITPACK_JSONL = ROOT_DIR / 'public_commitpack.jsonl'
NEW_LIBRARY_JSONL = ROOT_DIR / 'new_library_qwen.jsonl'
REJECTED_JSONL = ROOT_DIR / 'rejected.jsonl'
STATE_DB = ROOT_DIR / 'dataset_state.sqlite3'
FINAL_JSONL = ROOT_DIR / 'python_specialist_final.jsonl'
FINAL_CSV = ROOT_DIR / 'python_specialist_final.csv'
FINAL_PARQUET = ROOT_DIR / 'python_specialist_final.parquet'
FINAL_SUMMARY = ROOT_DIR / 'python_specialist_summary.json'

print('Root:', ROOT_DIR)

# Cell 5 - Final 24-column schema

Every source is normalized into this exact schema. `target_ir` is what CodeT5+ will eventually learn to emit; `target_code` remains the canonical Python target for verification and IR regeneration.

In [ ]:
DATASET_COLUMNS = [
    'id','task','instruction','code_before','error','project_context',
    'library_context','tests','target_mode','target_ir','target_code',
    'domain','subdomain','difficulty','instruction_style','concepts',
    'libraries','verified','verification_level','source_type','source_dataset',
    'source_reference','ir_version','content_hash',
]
assert len(DATASET_COLUMNS) == 24
for i, column in enumerate(DATASET_COLUMNS, 1):
    print(f'{i:02d}. {column}')

# Cell 6 - Python-oriented IR v1

The target is **not JSON** and not a raw CPython AST dump. It uses stable structural tags. Arbitrary identifiers, libraries, method names, and literals remain normal text.

For Repair/Refactor v1, the decoder predicts the **complete corrected target program IR** while the old source is supplied to the encoder. This gives a clean first baseline. Fine-grained edit IR can be tested later without changing the raw dataset because both `code_before` and `target_code` are retained.

In [ ]:
def safe_unparse(node: ast.AST | None) -> str:
    if node is None:
        return ''
    try:
        return ast.unparse(node)
    except Exception:
        return node.__class__.__name__


def scalar(value: Any) -> str:
    if isinstance(value, str):
        return json.dumps(value, ensure_ascii=False)
    return repr(value)


class PythonIRBuilder:
    def __init__(self):
        self.lines = []

    def add(self, line: str):
        self.lines.append(line)

    def build(self, source: str, mode: str) -> str:
        tree = ast.parse(source)
        self.lines = []
        self.add('<BUILD>' if mode == 'build' else '<EDIT>')
        self.add('<MODULE>')
        for stmt in tree.body:
            self.stmt(stmt)
        self.add('<END_MODULE>')
        self.add('<END>')
        return '\n'.join(self.lines)

    def param(self, arg: ast.arg, kind='positional', default=None):
        self.add('<PARAM>')
        self.add(f'<PARAM_KIND> {kind}')
        self.add(f'<NAME> {arg.arg}')
        if arg.annotation is not None:
            self.add(f'<TYPE> {safe_unparse(arg.annotation)}')
        if default is not None:
            self.add('<DEFAULT>')
            self.expr(default)
            self.add('<END_DEFAULT>')
        self.add('<END_PARAM>')

    def function_body(self, node):
        self.add(f'<NAME> {node.name}')
        for dec in node.decorator_list:
            self.add('<DECORATOR>')
            self.expr(dec)
            self.add('<END_DECORATOR>')
        pos = list(node.args.posonlyargs) + list(node.args.args)
        defaults = [None] * (len(pos) - len(node.args.defaults)) + list(node.args.defaults)
        posonly = {a.arg for a in node.args.posonlyargs}
        for arg, default in zip(pos, defaults):
            self.param(arg, 'positional_only' if arg.arg in posonly else 'positional', default)
        if node.args.vararg:
            self.param(node.args.vararg, 'vararg')
        for arg, default in zip(node.args.kwonlyargs, node.args.kw_defaults):
            self.param(arg, 'keyword_only', default)
        if node.args.kwarg:
            self.param(node.args.kwarg, 'kwarg')
        if node.returns is not None:
            self.add(f'<RETURN_TYPE> {safe_unparse(node.returns)}')
        body = list(node.body)
        doc = ast.get_docstring(node, clean=False)
        if doc is not None and body and isinstance(body[0], ast.Expr):
            self.add(f'<DOCSTRING> {json.dumps(doc, ensure_ascii=False)}')
            body = body[1:]
        self.add('<BODY>')
        for child in body:
            self.stmt(child)
        self.add('<END_BODY>')

    def stmt(self, node: ast.stmt):
        if isinstance(node, ast.Import):
            self.add('<IMPORT>')
            for alias in node.names:
                name = alias.name + (f' as {alias.asname}' if alias.asname else '')
                self.add(f'<NAME> {name}')
            self.add('<END_IMPORT>'); return
        if isinstance(node, ast.ImportFrom):
            self.add('<IMPORT_FROM>')
            self.add(f"<MODULE_NAME> {'.' * node.level}{node.module or ''}")
            for alias in node.names:
                name = alias.name + (f' as {alias.asname}' if alias.asname else '')
                self.add(f'<NAME> {name}')
            self.add('<END_IMPORT_FROM>'); return
        if isinstance(node, ast.FunctionDef):
            self.add('<FUNCTION>'); self.function_body(node); self.add('<END_FUNCTION>'); return
        if isinstance(node, ast.AsyncFunctionDef):
            self.add('<ASYNC_FUNCTION>'); self.function_body(node); self.add('<END_ASYNC_FUNCTION>'); return
        if isinstance(node, ast.ClassDef):
            self.add('<CLASS>'); self.add(f'<NAME> {node.name}')
            for base in node.bases:
                self.add('<BASE>'); self.expr(base); self.add('<END_BASE>')
            for dec in node.decorator_list:
                self.add('<DECORATOR>'); self.expr(dec); self.add('<END_DECORATOR>')
            body = list(node.body)
            doc = ast.get_docstring(node, clean=False)
            if doc is not None and body and isinstance(body[0], ast.Expr):
                self.add(f'<DOCSTRING> {json.dumps(doc, ensure_ascii=False)}'); body = body[1:]
            self.add('<BODY>')
            for child in body: self.stmt(child)
            self.add('<END_BODY>'); self.add('<END_CLASS>'); return
        if isinstance(node, ast.Assign):
            self.add('<ASSIGN>')
            for target in node.targets:
                self.add('<TARGET>'); self.expr(target); self.add('<END_TARGET>')
            self.add('<VALUE>'); self.expr(node.value); self.add('<END_VALUE>'); self.add('<END_ASSIGN>'); return
        if isinstance(node, ast.AnnAssign):
            self.add('<ANN_ASSIGN>'); self.add('<TARGET>'); self.expr(node.target); self.add('<END_TARGET>')
            self.add(f'<TYPE> {safe_unparse(node.annotation)}')
            if node.value is not None:
                self.add('<VALUE>'); self.expr(node.value); self.add('<END_VALUE>')
            self.add('<END_ANN_ASSIGN>'); return
        if isinstance(node, ast.AugAssign):
            self.add('<AUG_ASSIGN>'); self.add(f'<OP> {node.op.__class__.__name__}')
            self.add('<TARGET>'); self.expr(node.target); self.add('<END_TARGET>')
            self.add('<VALUE>'); self.expr(node.value); self.add('<END_VALUE>'); self.add('<END_AUG_ASSIGN>'); return
        if isinstance(node, ast.Return):
            self.add('<RETURN>')
            if node.value is not None: self.expr(node.value)
            self.add('<END_RETURN>'); return
        if isinstance(node, ast.Expr):
            if isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
                self.add(f'<DOCSTRING> {json.dumps(node.value.value, ensure_ascii=False)}')
            else:
                self.add('<EXPR_STMT>'); self.expr(node.value); self.add('<END_EXPR_STMT>')
            return
        if isinstance(node, ast.If):
            self.add('<IF>'); self.add('<CONDITION>'); self.expr(node.test); self.add('<END_CONDITION>')
            self.add('<BODY>'); [self.stmt(x) for x in node.body]; self.add('<END_BODY>')
            if node.orelse:
                self.add('<ELSE>'); [self.stmt(x) for x in node.orelse]; self.add('<END_ELSE>')
            self.add('<END_IF>'); return
        if isinstance(node, (ast.For, ast.AsyncFor)):
            a = isinstance(node, ast.AsyncFor); self.add('<ASYNC_FOR>' if a else '<FOR>')
            self.add('<TARGET>'); self.expr(node.target); self.add('<END_TARGET>')
            self.add('<ITER>'); self.expr(node.iter); self.add('<END_ITER>')
            self.add('<BODY>'); [self.stmt(x) for x in node.body]; self.add('<END_BODY>')
            if node.orelse:
                self.add('<ELSE>'); [self.stmt(x) for x in node.orelse]; self.add('<END_ELSE>')
            self.add('<END_ASYNC_FOR>' if a else '<END_FOR>'); return
        if isinstance(node, ast.While):
            self.add('<WHILE>'); self.add('<CONDITION>'); self.expr(node.test); self.add('<END_CONDITION>')
            self.add('<BODY>'); [self.stmt(x) for x in node.body]; self.add('<END_BODY>')
            if node.orelse:
                self.add('<ELSE>'); [self.stmt(x) for x in node.orelse]; self.add('<END_ELSE>')
            self.add('<END_WHILE>'); return
        if isinstance(node, (ast.With, ast.AsyncWith)):
            a = isinstance(node, ast.AsyncWith); self.add('<ASYNC_WITH>' if a else '<WITH>')
            for item in node.items:
                self.add('<WITH_ITEM>'); self.expr(item.context_expr)
                if item.optional_vars is not None:
                    self.add('<AS>'); self.expr(item.optional_vars); self.add('<END_AS>')
                self.add('<END_WITH_ITEM>')
            self.add('<BODY>'); [self.stmt(x) for x in node.body]; self.add('<END_BODY>')
            self.add('<END_ASYNC_WITH>' if a else '<END_WITH>'); return
        if isinstance(node, ast.Try):
            self.add('<TRY>'); self.add('<BODY>'); [self.stmt(x) for x in node.body]; self.add('<END_BODY>')
            for h in node.handlers:
                self.add('<EXCEPT>')
                if h.type is not None:
                    self.add('<EXCEPTION_TYPE>'); self.expr(h.type); self.add('<END_EXCEPTION_TYPE>')
                if h.name: self.add(f'<NAME> {h.name}')
                self.add('<BODY>'); [self.stmt(x) for x in h.body]; self.add('<END_BODY>'); self.add('<END_EXCEPT>')
            if node.orelse:
                self.add('<ELSE>'); [self.stmt(x) for x in node.orelse]; self.add('<END_ELSE>')
            if node.finalbody:
                self.add('<FINALLY>'); [self.stmt(x) for x in node.finalbody]; self.add('<END_FINALLY>')
            self.add('<END_TRY>'); return
        if isinstance(node, ast.Raise):
            self.add('<RAISE>')
            if node.exc is not None: self.expr(node.exc)
            self.add('<END_RAISE>'); return
        if isinstance(node, ast.Assert):
            self.add('<ASSERT>'); self.expr(node.test); self.add('<END_ASSERT>'); return
        if isinstance(node, ast.Pass): self.add('<PASS>'); return
        if isinstance(node, ast.Break): self.add('<BREAK>'); return
        if isinstance(node, ast.Continue): self.add('<CONTINUE>'); return
        self.add(f'<PY_STMT> {json.dumps(safe_unparse(node), ensure_ascii=False)}')

    def expr(self, node: ast.expr):
        if isinstance(node, ast.Name): self.add(f'<NAME_REF> {node.id}'); return
        if isinstance(node, ast.Constant): self.add(f'<CONST> {scalar(node.value)}'); return
        if isinstance(node, ast.Attribute):
            self.add('<ATTRIBUTE>'); self.add('<BASE>'); self.expr(node.value); self.add('<END_BASE>')
            self.add(f'<ATTR_NAME> {node.attr}'); self.add('<END_ATTRIBUTE>'); return
        if isinstance(node, ast.Call):
            self.add('<CALL>'); self.add('<FUNC>'); self.expr(node.func); self.add('<END_FUNC>')
            for arg in node.args:
                self.add('<ARG>'); self.expr(arg); self.add('<END_ARG>')
            for kw in node.keywords:
                self.add('<KWARG>'); self.add(f"<KW_NAME> {kw.arg if kw.arg is not None else '**'}")
                self.expr(kw.value); self.add('<END_KWARG>')
            self.add('<END_CALL>'); return
        if isinstance(node, ast.Await):
            self.add('<AWAIT>'); self.expr(node.value); self.add('<END_AWAIT>'); return
        if isinstance(node, ast.BinOp):
            self.add(f'<BINOP> {node.op.__class__.__name__}'); self.add('<LEFT>'); self.expr(node.left); self.add('<END_LEFT>')
            self.add('<RIGHT>'); self.expr(node.right); self.add('<END_RIGHT>'); self.add('<END_BINOP>'); return
        if isinstance(node, ast.BoolOp):
            self.add(f'<BOOL_OP> {node.op.__class__.__name__}')
            for value in node.values:
                self.add('<VALUE>'); self.expr(value); self.add('<END_VALUE>')
            self.add('<END_BOOL_OP>'); return
        if isinstance(node, ast.UnaryOp):
            self.add(f'<UNARY_OP> {node.op.__class__.__name__}'); self.expr(node.operand); self.add('<END_UNARY_OP>'); return
        if isinstance(node, ast.Compare):
            self.add('<COMPARE>'); self.add('<LEFT>'); self.expr(node.left); self.add('<END_LEFT>')
            for op, comp in zip(node.ops, node.comparators):
                self.add(f'<COMPARE_OP> {op.__class__.__name__}'); self.expr(comp); self.add('<END_COMPARE_OP>')
            self.add('<END_COMPARE>'); return
        if isinstance(node, ast.Subscript):
            self.add('<SUBSCRIPT>'); self.add('<VALUE>'); self.expr(node.value); self.add('<END_VALUE>')
            self.add('<SLICE>'); self.expr(node.slice); self.add('<END_SLICE>'); self.add('<END_SUBSCRIPT>'); return
        if isinstance(node, ast.List):
            self.add('<LIST>'); [self.expr(x) for x in node.elts]; self.add('<END_LIST>'); return
        if isinstance(node, ast.Tuple):
            self.add('<TUPLE>'); [self.expr(x) for x in node.elts]; self.add('<END_TUPLE>'); return
        if isinstance(node, ast.Set):
            self.add('<SET>'); [self.expr(x) for x in node.elts]; self.add('<END_SET>'); return
        if isinstance(node, ast.Dict):
            self.add('<DICT>')
            for key, value in zip(node.keys, node.values):
                self.add('<ITEM>')
                if key is not None:
                    self.add('<KEY>'); self.expr(key); self.add('<END_KEY>')
                else: self.add('<DICT_UNPACK>')
                self.add('<VALUE>'); self.expr(value); self.add('<END_VALUE>'); self.add('<END_ITEM>')
            self.add('<END_DICT>'); return
        self.add(f'<PY_EXPR> {json.dumps(safe_unparse(node), ensure_ascii=False)}')


def python_to_ir(source: str, mode: str) -> str:
    if mode not in {'build', 'edit'}:
        raise ValueError(mode)
    return PythonIRBuilder().build(source, mode)

print(python_to_ir('async def fetch(client, url):\n    return await client.get(url)\n', 'edit'))

# Cell 7 - Full schema preview before loading Qwen

This is a static schema preview, not teacher-generated data. It immediately shows what complete rows look like. The cell prints all 24 columns with no DataFrame column hiding.

In [ ]:
STARTER_EXAMPLE_ROWS = [
    {
        'id':'preview_gen_001','task':'generate',
        'instruction':'Create a typed function that preserves order while removing duplicate strings.',
        'code_before':'','error':'','project_context':'','library_context':'',
        'tests':"from solution import unique_strings\n\ndef test_unique_strings():\n    assert unique_strings(['a','b','a']) == ['a','b']\n",
        'target_mode':'build','target_ir':'<BUILD>\n<MODULE>\n<FUNCTION> ... <END_FUNCTION>\n<END_MODULE>\n<END>',
        'target_code':"def unique_strings(values: list[str]) -> list[str]:\n    return list(dict.fromkeys(values))\n",
        'domain':'core_python','subdomain':'collections','difficulty':'beginner','instruction_style':'normal',
        'concepts':'["typing","deduplication"]','libraries':'[]','verified':True,
        'verification_level':'structural','source_type':'schema_preview','source_dataset':'','source_reference':'',
        'ir_version':'v1','content_hash':'preview_gen'
    },
    {
        'id':'preview_repair_001','task':'repair','instruction':'Fix this.',
        'code_before':'async def fetch(client, url):\n    return client.get(url)\n',
        'error':'RuntimeWarning: coroutine was never awaited','project_context':'The project is asynchronous.','library_context':'',
        'tests':'','target_mode':'edit','target_ir':'<EDIT>\n<MODULE>\n<ASYNC_FUNCTION> ... <AWAIT> ... <END_AWAIT> ...\n<END_MODULE>\n<END>',
        'target_code':'async def fetch(client, url):\n    return await client.get(url)\n',
        'domain':'asyncio','subdomain':'async_await','difficulty':'intermediate','instruction_style':'vague',
        'concepts':'["asyncio","await"]','libraries':'[]','verified':True,'verification_level':'structural',
        'source_type':'schema_preview','source_dataset':'','source_reference':'','ir_version':'v1','content_hash':'preview_repair'
    },
    {
        'id':'preview_newlib_001','task':'generate','instruction':'Cache users for 60 seconds using the supplied SDK.',
        'code_before':'','error':'','project_context':'',
        'library_context':'Package: zephyr_cache\nclass CacheBridge:\n    async def read_entry(self, key: str) -> bytes | None: ...\n    async def write_entry(self, key: str, value: bytes, ttl_seconds: int) -> None: ...',
        'tests':'','target_mode':'build','target_ir':'<BUILD>\n<MODULE> ... <END_MODULE>\n<END>',
        'target_code':"from zephyr_cache import CacheBridge\n\nasync def cache_user(cache: CacheBridge, key: str, value: bytes) -> None:\n    await cache.write_entry(key, value, ttl_seconds=60)\n",
        'domain':'software_design','subdomain':'new_library_integration','difficulty':'intermediate','instruction_style':'normal',
        'concepts':'["unfamiliar API","asyncio"]','libraries':'["zephyr_cache"]','verified':True,
        'verification_level':'structural','source_type':'schema_preview','source_dataset':'synthetic_library_context',
        'source_reference':'','ir_version':'v1','content_hash':'preview_newlib'
    },
]
preview_df = pd.DataFrame(STARTER_EXAMPLE_ROWS)[DATASET_COLUMNS]
print(preview_df.to_string(index=False))

# Cell 8 - Validation, metadata, and code cleaning

In [ ]:
CODE_FENCE_RE = re.compile(r'```(?:python|py)?\s*(.*?)```', re.I | re.S)
PLACEHOLDER_PATTERNS = [r'\bTODO\b', r'<implementation>', r'<complete code>', r'#\s*rest omitted', r'#\s*implementation omitted']


def clean_code(text: str) -> str:
    text = (text or '').strip()
    fenced = CODE_FENCE_RE.findall(text)
    if fenced:
        text = max(fenced, key=len).strip()
    text = re.sub(r'^```(?:python|py)?\s*', '', text, flags=re.I)
    text = re.sub(r'\s*```$', '', text)
    for prefix in ['Here is the code:', "Here's the code:", 'Here is the Python code:', "Here's the Python code:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].lstrip()
    return text.rstrip() + ('\n' if text else '')


def parse_and_compile(source: str):
    if not source.strip(): return False, 'empty source'
    try:
        tree = ast.parse(source); compile(tree, '<dataset-example>', 'exec'); return True, ''
    except Exception as exc:
        return False, f'{type(exc).__name__}: {exc}'


def has_placeholder(source: str) -> bool:
    return any(re.search(p, source, flags=re.I) for p in PLACEHOLDER_PATTERNS)


def extract_imports(source: str):
    try: tree = ast.parse(source)
    except Exception: return []
    mods = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            mods.update(a.name.split('.')[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods.add(node.module.split('.')[0])
    return sorted(mods)

STDLIB_MODULES = set(getattr(sys, 'stdlib_module_names', set()))

def extract_third_party_libraries(source: str):
    return [m for m in extract_imports(source) if m not in STDLIB_MODULES and m != '__future__']


def ast_complexity_score(source: str) -> int:
    tree = ast.parse(source); score = 0
    for node in ast.walk(tree):
        score += 1
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.Try, ast.With, ast.AsyncWith, ast.Match)): score += 4
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): score += 5
        if isinstance(node, ast.comprehension): score += 2
    return score


def infer_difficulty(source: str) -> str:
    score = ast_complexity_score(source) + len(source.splitlines())
    if score < 45: return 'beginner'
    if score < 140: return 'intermediate'
    if score < 320: return 'advanced'
    return 'expert'


def infer_instruction_style(instruction: str) -> str:
    n = len((instruction or '').split())
    if n <= 5: return 'vague'
    if n <= 30: return 'normal'
    if n <= 90: return 'detailed'
    return 'expert_constrained'


def infer_domain(source: str, instruction: str = ''):
    text = (source + '\n' + instruction).lower(); imports = set(extract_imports(source))
    if imports & {'torch','tensorflow','keras','jax'}: return 'machine_learning','modeling'
    if imports & {'numpy','pandas','scipy','sklearn','matplotlib'}: return 'data_science','data_processing'
    if imports & {'sqlalchemy','sqlite3','asyncpg','psycopg','pymongo'}: return 'database','database_access'
    if imports & {'fastapi','flask','django','starlette'}: return 'api','web_api'
    if imports & {'httpx','requests','aiohttp'}: return 'networking','http_client'
    if 'async ' in text or 'await ' in text or 'asyncio' in imports: return 'asyncio','async_await'
    if imports & {'pytest','unittest'}: return 'testing','test_automation'
    if imports & {'cryptography','jwt','bcrypt','secrets'} or 'authorization' in text: return 'security','application_security'
    if any(x in text for x in ['repository','strategy','factory','dependency injection']): return 'software_design','design_patterns'
    return 'core_python','general'


def infer_concepts(source: str, instruction: str = ''):
    text = (source + '\n' + instruction).lower(); out=[]
    checks = {
        'asyncio':['async ','await ','asyncio'],'typing':['->',': list[','typing','protocol','typevar'],
        'testing':['pytest','unittest','assert '],'database':['sqlalchemy','transaction','commit(','rollback('],
        'api_design':['fastapi','flask','django','router.','app.get','app.post'],
        'authentication':['authenticate','login','token'],'authorization':['authorize','permission','role'],
        'security':['hashlib','bcrypt','cryptography','sanitize','validate'],
        'concurrency':['thread','process','semaphore','lock','taskgroup'],
        'error_handling':['try:','except ','raise '],'resource_management':['with ','async with','close('],
        'data_processing':['pandas','numpy','dataframe'],'machine_learning':['torch','tensorflow','sklearn'],
    }
    for concept, needles in checks.items():
        if any(n in text for n in needles): out.append(concept)
    return out[:12]


def normalized_text(text: str) -> str:
    text = (text or '').replace('\r\n','\n').strip()
    text = re.sub(r'[ \t]+',' ',text); text = re.sub(r'\n{3,}','\n\n',text)
    return text


def make_content_hash(row):
    payload = '\n---\n'.join(normalized_text(str(row.get(k,''))) for k in [
        'task','instruction','code_before','error','project_context','library_context','target_code'])
    return hashlib.sha256(payload.encode()).hexdigest()


def make_structure_hash(source: str) -> str:
    try: tree = ast.parse(source)
    except Exception: return ''
    class Norm(ast.NodeTransformer):
        def visit_Name(self,node): return ast.copy_location(ast.Name(id='<NAME>',ctx=node.ctx),node)
        def visit_arg(self,node): node.arg='<ARG>'; return self.generic_visit(node)
        def visit_Constant(self,node):
            if isinstance(node.value,(str,int,float,bytes)): return ast.copy_location(ast.Constant(value='<CONST>'),node)
            return node
    tree=Norm().visit(tree); ast.fix_missing_locations(tree)
    return hashlib.sha256(ast.dump(tree,annotate_fields=True,include_attributes=False).encode()).hexdigest()

print('Utilities ready.')

# Cell 9 - Final row builder

In [ ]:
def normalize_json_list(value: Any) -> str:
    if value is None: items=[]
    elif isinstance(value,str):
        try:
            loaded=json.loads(value); items=loaded if isinstance(loaded,list) else [value]
        except Exception: items=[value] if value.strip() else []
    else: items=list(value)
    clean=[]; seen=set()
    for item in items:
        s=str(item).strip()
        if s and s not in seen: clean.append(s); seen.add(s)
    return json.dumps(clean,ensure_ascii=False,separators=(',',':'))


def build_dataset_row(*, task, instruction, target_code, code_before='', error='', project_context='',
                      library_context='', tests='', domain=None, subdomain=None, difficulty=None,
                      instruction_style=None, concepts=None, libraries=None, verified=True,
                      verification_level='structural', source_type='', source_dataset='', source_reference=''):
    if task not in TASKS: raise ValueError(f'Unknown task: {task}')
    target_code=clean_code(target_code); code_before=clean_code(code_before)
    target_mode='build' if task in {'generate','full_file_generate'} else 'edit'
    ok,msg=parse_and_compile(target_code)
    if not ok: raise ValueError(f'Target invalid: {msg}')
    if has_placeholder(target_code): raise ValueError('Target contains placeholder text')
    if target_mode=='build' and code_before.strip(): raise ValueError('Build task has code_before')
    if target_mode=='edit' and not code_before.strip(): raise ValueError('Edit task requires code_before')
    if task=='refactor':
        ok,msg=parse_and_compile(code_before)
        if not ok: raise ValueError(f'Refactor before invalid: {msg}')
    if task=='repair':
        ok,msg=parse_and_compile(code_before)
        if not ok and 'SyntaxError' not in (error or ''): raise ValueError(f'Repair before invalid: {msg}')
    inferred_domain,inferred_subdomain=infer_domain(target_code,instruction)
    row={
        'id':'','task':task,'instruction':(instruction or '').strip(),'code_before':code_before,
        'error':(error or '').strip(),'project_context':(project_context or '').strip(),
        'library_context':(library_context or '').strip(),'tests':(tests or '').strip(),
        'target_mode':target_mode,'target_ir':python_to_ir(target_code,target_mode),'target_code':target_code,
        'domain':domain or inferred_domain,'subdomain':subdomain or inferred_subdomain,
        'difficulty':difficulty or infer_difficulty(target_code),
        'instruction_style':instruction_style or infer_instruction_style(instruction),
        'concepts':normalize_json_list(concepts or infer_concepts(target_code,instruction)),
        'libraries':normalize_json_list(libraries or extract_third_party_libraries(target_code)),
        'verified':bool(verified),'verification_level':verification_level,'source_type':source_type,
        'source_dataset':source_dataset,'source_reference':source_reference,'ir_version':IR_VERSION,'content_hash':''}
    if not row['instruction']: raise ValueError('Instruction cannot be empty')
    row['content_hash']=make_content_hash(row); row['id']=f"pys_{task}_{row['content_hash'][:14]}"
    return row

print('Row builder ready.')

# Cell 10 - Resumable storage and global deduplication

Each source has a JSONL shard. SQLite indexes accepted hashes so generation can resume after a Colab disconnect.

In [ ]:
SOURCE_FILES=[TEACHER_JSONL,MAGICODER_JSONL,COMMITPACK_JSONL,NEW_LIBRARY_JSONL]

def append_jsonl(path: Path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a',encoding='utf-8') as f: f.write(json.dumps(obj,ensure_ascii=False)+'\n')

def read_jsonl(path: Path):
    if not path.exists(): return []
    rows=[]
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

class DatasetStore:
    def __init__(self,path):
        self.conn=sqlite3.connect(str(path))
        self.conn.execute('CREATE TABLE IF NOT EXISTS accepted(content_hash TEXT PRIMARY KEY, structure_hash TEXT, row_id TEXT, source_type TEXT, task TEXT)')
        self.conn.commit()
    def rebuild(self,files):
        for path in files:
            for row in read_jsonl(path):
                self.conn.execute('INSERT OR IGNORE INTO accepted VALUES(?,?,?,?,?)',(
                    row['content_hash'],make_structure_hash(row.get('target_code','')),row['id'],row.get('source_type',''),row.get('task','')))
        self.conn.commit()
    def has_content(self,h):
        return self.conn.execute('SELECT 1 FROM accepted WHERE content_hash=? LIMIT 1',(h,)).fetchone() is not None
    def has_structure(self,h):
        return bool(h) and self.conn.execute('SELECT 1 FROM accepted WHERE structure_hash=? LIMIT 1',(h,)).fetchone() is not None
    def add(self,row,path):
        if self.has_content(row['content_hash']): return False,'duplicate_content'
        sh=make_structure_hash(row['target_code'])
        if STRICT_STRUCTURE_DEDUP and self.has_structure(sh): return False,'duplicate_structure'
        append_jsonl(path,row)
        self.conn.execute('INSERT INTO accepted VALUES(?,?,?,?,?)',(row['content_hash'],sh,row['id'],row['source_type'],row['task']))
        self.conn.commit(); return True,'accepted'
    def counts(self):
        return pd.read_sql_query('SELECT source_type,task,COUNT(*) AS rows FROM accepted GROUP BY source_type,task ORDER BY source_type,task',self.conn)

store=DatasetStore(STATE_DB); store.rebuild(SOURCE_FILES)
display(store.counts())

# Section A - Qwen Teacher Synthesis

The teacher creates the semantic content: realistic instructions, project context, broken/working code, errors, tests, target Python, concepts, and libraries.

Deterministic code creates IDs, hashes, metadata normalization, validation, and the IR.

# Cell 11 - Verify T4 GPU

In [ ]:
assert torch.cuda.is_available(), 'No CUDA GPU detected. Select a T4 runtime in Colab.'
gpu_name=torch.cuda.get_device_name(0)
gpu_mem=torch.cuda.get_device_properties(0).total_memory/1024**3
print('GPU:',gpu_name)
print(f'VRAM: {gpu_mem:.2f} GB')

# Cell 12 - Load Qwen2.5-Coder-7B-Instruct in 4-bit

The official checkpoint is quantized at load time with NF4 so it fits a T4-class GPU.

In [ ]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
teacher_tokenizer=AutoTokenizer.from_pretrained(TEACHER_MODEL_ID,use_fast=True)
if teacher_tokenizer.pad_token_id is None:
    teacher_tokenizer.pad_token_id=teacher_tokenizer.eos_token_id
teacher_tokenizer.padding_side='left'
started=time.perf_counter()
teacher_model=AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
teacher_model.eval()
print(f'Loaded in {time.perf_counter()-started:.1f}s')
print(f'CUDA allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

# Cell 13 - Detailed teacher prompt

Priority taught to Qwen:

1. Explicit user requirement.
2. Existing project/framework/library conventions.
3. Supplied library API documentation.
4. Sensible model default only where the request leaves freedom.
5. Simplicity over unnecessary abstraction.

For unfamiliar libraries, the supplied signatures are authoritative and Qwen must not invent methods.

In [ ]:
TEACHER_SYSTEM_PROMPT = """
You are the teacher model for a Python-specialist training-data factory.

Create high-quality Python software-engineering examples for an encoder-decoder student.

BEHAVIORAL PRIORITY:
1. Follow explicit user requirements.
2. Preserve existing project/framework/library conventions.
3. Respect supplied API signatures and library documentation exactly.
4. Make sensible engineering decisions only where context leaves freedom.
5. Prefer simple maintainable solutions over unnecessary abstractions.

QUALITY EXPECTATIONS:
- correct, idiomatic, maintainable Python
- meaningful typing where appropriate
- good error/resource handling
- security and testability
- appropriate SOLID/design patterns, never ritualistically
- sound API/database semantics
- correct async/concurrency behavior
- real APIs only
- no placeholders, TODOs, omitted code, or fake APIs

OUTPUT CONTRACT:
Return exactly ONE valid JSON object and nothing else.
Do not use Markdown fences.
Required keys:
{
  "instruction": "realistic user request",
  "code_before": "",
  "error": "",
  "project_context": "",
  "tests": "pytest source when practical",
  "target_code": "complete actual Python source",
  "concepts": ["..."],
  "libraries": ["..."]
}

When practical tests should import from a module named `solution`.
Generate/full-file tests pass against target_code.
Repair tests expose the bug before and pass after.
Refactor tests pass before and after.
"""

STYLE_GUIDANCE={
    'vague':"Use a deliberately short request such as 'Fix this.' or 'Refactor this.' while code/context still makes the task solvable.",
    'normal':'Write a realistic everyday developer request and leave reasonable implementation choices open.',
    'detailed':'Write a detailed developer request with several functional constraints.',
    'expert_constrained':'Write a precise expert requirement with meaningful architecture, correctness, performance, security, API, DB, or concurrency constraints.',
}


def build_teacher_prompt(spec):
    task=spec['task']; library_context=spec.get('library_context',''); project_context=spec.get('project_context','')
    common=f"""
Create ONE unique training example.

TASK: {task}
DOMAIN: {spec.get('domain','core_python')}
SUBDOMAIN: {spec.get('subdomain','general')}
DIFFICULTY: {spec.get('difficulty','intermediate')}
INSTRUCTION STYLE: {spec.get('instruction_style','normal')}

Instruction style guidance:
{STYLE_GUIDANCE[spec.get('instruction_style','normal')]}

Project context supplied by controller:
{project_context or '(none)'}

Library/API context supplied by controller:
{library_context or '(none)'}

Additional requirements:
{spec.get('extra_requirements','') or '(none)'}
"""
    if library_context:
        common += """
UNFAMILIAR-LIBRARY RULES:
- Treat supplied API context as authoritative.
- Do not invent methods/classes/arguments not stated there.
- Target code must actually use the supplied package when required.
- Combine the API surface with general software-engineering knowledge.
"""
    rules={
        'generate':"""code_before and error must be empty. target_code must be a complete usable implementation. Do not over-engineer a small request.""",
        'full_file_generate':"""code_before and error must be empty. target_code must be the COMPLETE contents of a realistic importable .py module with all required imports and no omitted sections.""",
        'repair':"""code_before must contain actual broken Python and target_code the corrected Python. Prefer semantic/runtime/concurrency/API/design bugs over trivial syntax mistakes. error may be empty for a vague 'Fix this.' request.""",
        'refactor':"""code_before must contain working Python and target_code a behavior-preserving refactor. Improve only justified aspects and do not add patterns merely to look sophisticated.""",
    }[task]
    return common + '\nTASK RULES:\n' + rules + '\nReturn the required JSON object only.'

print(build_teacher_prompt({'task':'repair','domain':'asyncio','subdomain':'cancellation','difficulty':'advanced','instruction_style':'vague'}))

# Cell 14 - Qwen generation and JSON parsing

No `max_new_tokens` is passed. The only limit is total sequence length (`GENERATION_MAX_TOTAL_TOKENS`).

In [ ]:
def extract_first_json_object(text: str):
    text=text.strip(); text=re.sub(r'^```(?:json)?\s*','',text,flags=re.I); text=re.sub(r'\s*```$','',text)
    try:
        obj=json.loads(text)
        if isinstance(obj,dict): return obj
    except Exception: pass
    decoder=json.JSONDecoder()
    for i,ch in enumerate(text):
        if ch!='{': continue
        try:
            obj,_=decoder.raw_decode(text[i:])
            if isinstance(obj,dict): return obj
        except json.JSONDecodeError: pass
    raise ValueError('No valid JSON object found in Qwen output')

@torch.inference_mode()
def qwen_generate_json(prompt: str, temperature=0.65, top_p=0.90):
    messages=[{'role':'system','content':TEACHER_SYSTEM_PROMPT},{'role':'user','content':prompt}]
    encoded=teacher_tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True)
    encoded={k:v.to(teacher_model.device) for k,v in encoded.items()}
    input_tokens=encoded['input_ids'].shape[-1]
    if input_tokens>=GENERATION_MAX_TOTAL_TOKENS-128:
        raise ValueError(f'Prompt too long: {input_tokens} tokens')
    output=teacher_model.generate(
        **encoded,
        max_length=GENERATION_MAX_TOTAL_TOKENS,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.04,
        eos_token_id=teacher_tokenizer.eos_token_id,
        pad_token_id=teacher_tokenizer.pad_token_id,
        use_cache=True,
    )
    text=teacher_tokenizer.decode(output[0,input_tokens:],skip_special_tokens=True).strip()
    return extract_first_json_object(text)

print('Qwen generation ready.')

# Cell 15 - Validate Qwen output and convert to final schema

In [ ]:
TEACHER_REQUIRED_KEYS={'instruction','code_before','error','project_context','tests','target_code','concepts','libraries'}

def normalize_teacher_object(obj):
    missing=TEACHER_REQUIRED_KEYS-set(obj)
    if missing: raise ValueError(f'Missing keys: {sorted(missing)}')
    return {
        'instruction':str(obj.get('instruction') or '').strip(),
        'code_before':clean_code(str(obj.get('code_before') or '')),
        'error':str(obj.get('error') or '').strip(),
        'project_context':str(obj.get('project_context') or '').strip(),
        'tests':clean_code(str(obj.get('tests') or '')),
        'target_code':clean_code(str(obj.get('target_code') or '')),
        'concepts':obj.get('concepts') or [],
        'libraries':obj.get('libraries') or [],
    }

def teacher_object_to_row(obj,spec,source_type='qwen_teacher_synthetic'):
    obj=normalize_teacher_object(obj)
    if obj['tests'].strip():
        ok,msg=parse_and_compile(obj['tests'])
        if not ok: raise ValueError(f'Tests do not compile: {msg}')
    row=build_dataset_row(
        task=spec['task'],instruction=obj['instruction'],code_before=obj['code_before'],error=obj['error'],
        project_context=obj['project_context'] or spec.get('project_context',''),
        library_context=spec.get('library_context',''),tests=obj['tests'],target_code=obj['target_code'],
        domain=spec.get('domain'),subdomain=spec.get('subdomain'),difficulty=spec.get('difficulty'),
        instruction_style=spec.get('instruction_style'),concepts=obj['concepts'],libraries=obj['libraries'],
        verified=True,verification_level='structural',source_type=source_type,source_dataset=TEACHER_MODEL_ID,
        source_reference=spec.get('source_reference',''))
    package=spec.get('library_package'); required=set(spec.get('must_use_symbols',[]))
    if package:
        if package not in row['target_code']: raise ValueError(f'Target does not reference required package {package}')
        if required and not any(x in row['target_code'] for x in required): raise ValueError(f'Target uses none of required symbols: {sorted(required)}')
        libs=json.loads(row['libraries'])
        if package not in libs: libs.append(package); row['libraries']=normalize_json_list(libs)
    return row

def generate_teacher_row(spec,source_type='qwen_teacher_synthetic',retries=TEACHER_RETRIES_PER_ROW):
    errors=[]
    for attempt in range(1,retries+1):
        try:
            obj=qwen_generate_json(build_teacher_prompt(spec))
            return teacher_object_to_row(obj,spec,source_type),errors
        except Exception as exc:
            errors.append(f'attempt {attempt}: {type(exc).__name__}: {exc}')
            gc.collect(); torch.cuda.empty_cache()
    return None,errors

print('Teacher candidate validation ready.')

# Cell 16 - Coverage controller

Teacher synthesis is spread across broad Python domains, four difficulty levels, and realistic prompt styles.

In [ ]:
DOMAIN_SUBDOMAINS={
 'core_python':['typing','collections','iterators','generators','decorators','context_managers','dataclasses','filesystem','serialization','cli','descriptors','exceptions'],
 'software_design':['solid','dependency_injection','repository_pattern','strategy_pattern','factory_pattern','adapter_pattern','separation_of_concerns','simplification'],
 'api':['rest_design','fastapi','flask','django','validation','pagination','idempotency','webhooks','middleware','error_contracts'],
 'database':['transactions','sqlalchemy','async_sqlalchemy','indexes','relationships','locking','optimistic_concurrency','query_optimization','unit_of_work'],
 'asyncio':['async_await','taskgroup','cancellation','timeouts','queues','bounded_concurrency','backpressure'],
 'concurrency':['threading','multiprocessing','locks','race_conditions','producer_consumer'],
 'security':['input_validation','authorization','authentication','path_traversal','ssrf','subprocess_safety','secret_handling'],
 'testing':['pytest','fixtures','mocking','async_testing','regression_testing'],
 'performance':['complexity','memory_efficiency','streaming','caching','batching'],
 'data_science':['numpy','pandas','scipy','scikit_learn','data_cleaning'],
 'machine_learning':['pytorch','dataloader','training_loop','inference','checkpointing'],
 'data_engineering':['etl','batching','streaming_data','deduplication','parquet'],
 'networking':['httpx','requests','retries','backoff','connection_pooling'],
 'packaging':['modules','packages','configuration','logging','application_lifecycle'],
}
DIFFICULTIES=['beginner']*15+['intermediate']*35+['advanced']*35+['expert']*15
STYLES=['vague']*20+['normal']*40+['detailed']*30+['expert_constrained']*10

def make_teacher_spec(task,index):
    rng=random.Random(SEED+index*101+sum(ord(c) for c in task))
    domain=rng.choice(list(DOMAIN_SUBDOMAINS)); subdomain=rng.choice(DOMAIN_SUBDOMAINS[domain])
    return {'task':task,'domain':domain,'subdomain':subdomain,'difficulty':rng.choice(DIFFICULTIES),
            'instruction_style':rng.choice(STYLES),'project_context':'','library_context':'',
            'extra_requirements':'Make this distinct from common benchmark-style toy problems and use realistic engineering behavior.',
            'source_reference':f'teacher:{task}:{index:08d}'}

for task in TASKS: print(task,'->',make_teacher_spec(task,0))

# Cell 17 - Real Qwen demo: one example for each task

This cell prints every field vertically and then prints the full 24-column DataFrame without truncation.

In [ ]:
demo_specs=[
 {'task':'generate','domain':'core_python','subdomain':'context_managers','difficulty':'intermediate','instruction_style':'normal','project_context':'','library_context':'','extra_requirements':'Use only the standard library.','source_reference':'demo:qwen:generate'},
 {'task':'full_file_generate','domain':'networking','subdomain':'retries','difficulty':'advanced','instruction_style':'detailed','project_context':'','library_context':'','extra_requirements':'Produce a realistic complete module.','source_reference':'demo:qwen:full'},
 {'task':'repair','domain':'asyncio','subdomain':'cancellation','difficulty':'advanced','instruction_style':'vague','project_context':'','library_context':'','extra_requirements':'Use a semantic async bug, not a syntax typo.','source_reference':'demo:qwen:repair'},
 {'task':'refactor','domain':'software_design','subdomain':'separation_of_concerns','difficulty':'advanced','instruction_style':'vague','project_context':'','library_context':'','extra_requirements':'Preserve externally visible behavior.','source_reference':'demo:qwen:refactor'},
]
qwen_demo_rows=[]
for spec in demo_specs:
    print('\n'+'='*120); print('GENERATING:',spec['task']); print('='*120)
    row,errors=generate_teacher_row(spec)
    if row is None:
        print('FAILED:',errors); continue
    qwen_demo_rows.append(row)
    for column in DATASET_COLUMNS:
        print(f'\n### {column}\n{row[column]}')
demo_df=pd.DataFrame(qwen_demo_rows,columns=DATASET_COLUMNS)
print('\nFULL DATAFRAME - ALL ROWS / ALL 24 COLUMNS')
print(demo_df.to_string(index=False))

# Cell 18 - Bulk Qwen teacher generation

Accepted rows are checkpointed immediately to `teacher_qwen.jsonl`; rejected attempts are logged separately.

In [ ]:
existing=read_jsonl(TEACHER_JSONL); counts=Counter(r['task'] for r in existing)
print('Existing counts:',counts)
accepted_total=0; rejected_total=0
for task in TASKS:
    needed=max(QWEN_TARGET_PER_TASK-counts.get(task,0),0)
    for local_i in tqdm(range(needed),desc=f'Qwen {task}'):
        idx=counts.get(task,0)+local_i; spec=make_teacher_spec(task,idx)
        row,errors=generate_teacher_row(spec)
        if row is None:
            rejected_total+=1; append_jsonl(REJECTED_JSONL,{'pipeline':'qwen_teacher','spec':spec,'errors':errors,'timestamp':time.time()}); continue
        accepted,reason=store.add(row,TEACHER_JSONL)
        if accepted: accepted_total+=1
        else:
            rejected_total+=1; append_jsonl(REJECTED_JSONL,{'pipeline':'qwen_teacher','spec':spec,'reason':reason,'timestamp':time.time()})
        if accepted_total and accepted_total%10==0: gc.collect(); torch.cuda.empty_cache()
print('Accepted:',accepted_total,'Rejected:',rejected_total)
display(store.counts())

# Section B - Open-Source Python Dataset Transformation

Two public sources are used because they complement each other.

## Magicoder OSS-Instruct 75K

Used for `generate` and `full_file_generate`.
Expected fields: `lang`, `problem`, `solution`, `raw_index`, `index`.

## CommitPackFT Python

Used for `repair` and `refactor`.
Expected fields: `old_contents`, `new_contents`, `subject`, `message`, `commit`, `old_file`, `new_file`, `license`, `repos`.

Only valid Python is accepted. Provenance is preserved. Review source licenses/terms before publishing a derived dataset.

# Cell 19 - Magicoder adapter

In [ ]:
MAGICODER_DATASET_ID='ise-uiuc/Magicoder-OSS-Instruct-75K'

def looks_like_complete_module(source: str) -> bool:
    try: tree=ast.parse(source)
    except Exception: return False
    defs=sum(isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef)) for n in tree.body)
    imports=sum(isinstance(n,(ast.Import,ast.ImportFrom)) for n in tree.body)
    return len(source.splitlines())>=40 and (defs>=2 or (defs>=1 and imports>=2))


def convert_magicoder_row(raw):
    if 'python' not in str(raw.get('lang','')).lower(): return None
    instruction=str(raw.get('problem') or '').strip(); target=clean_code(str(raw.get('solution') or ''))
    if not instruction or not target: return None
    if not MIN_CODE_CHARS<=len(target)<=MAX_CODE_CHARS: return None
    ok,_=parse_and_compile(target)
    if not ok or has_placeholder(target): return None
    task='full_file_generate' if looks_like_complete_module(target) else 'generate'
    domain,subdomain=infer_domain(target,instruction)
    ref=str(raw.get('raw_index') or raw.get('index') or raw.get('seed') or '')
    try:
        return build_dataset_row(task=task,instruction=instruction,target_code=target,domain=domain,subdomain=subdomain,
            difficulty=infer_difficulty(target),instruction_style=infer_instruction_style(instruction),
            concepts=infer_concepts(target,instruction),libraries=extract_third_party_libraries(target),
            verified=True,verification_level='structural',source_type='public_transformed',
            source_dataset=MAGICODER_DATASET_ID,source_reference=f'magicoder:{ref}')
    except Exception: return None

print('Magicoder adapter ready.')

# Cell 20 - Stream Magicoder and show full converted examples

In [ ]:
stream=load_dataset(MAGICODER_DATASET_ID,split='train',streaming=True)
preview=[]
for raw in stream:
    row=convert_magicoder_row(raw)
    if row is not None: preview.append(row)
    if len(preview)>=3: break
magicoder_preview_df=pd.DataFrame(preview,columns=DATASET_COLUMNS)
print(magicoder_preview_df.to_string(index=False))

# Cell 21 - Build Magicoder shard

In [ ]:
existing=len(read_jsonl(MAGICODER_JSONL)); needed=max(MAGICODER_ACCEPTED_TARGET-existing,0)
accepted_count=0; duplicate_count=0; scanned=0
print('Existing:',existing,'Need:',needed)
if needed:
    stream=load_dataset(MAGICODER_DATASET_ID,split='train',streaming=True)
    for raw in tqdm(stream,desc='Magicoder conversion'):
        scanned+=1; row=convert_magicoder_row(raw)
        if row is None: continue
        accepted,reason=store.add(row,MAGICODER_JSONL)
        if accepted: accepted_count+=1
        else: duplicate_count+=1
        if accepted_count>=needed: break
print('Scanned:',scanned,'Accepted:',accepted_count,'Dedup:',duplicate_count)
display(store.counts())

# Cell 22 - CommitPackFT adapter

Commit messages are conservatively classified. Ambiguous rows are skipped rather than forced into Repair or Refactor.

In [ ]:
COMMITPACK_PY_URL='https://huggingface.co/datasets/bigcode/commitpackft/resolve/main/data/python/data.jsonl'
OPEN_SOURCE_LICENSES={'mit','apache-2.0','bsd-3-clause','bsd-2-clause','isc','mpl-2.0','epl-1.0','lgpl-2.1','agpl-3.0','artistic-2.0','cc0-1.0','unlicense'}
REPAIR_WORDS={'fix','bug','broken','crash','error','failure','regression','incorrect','issue','leak','deadlock'}
REFACTOR_WORDS={'refactor','cleanup','clean-up','simplify','reorganize','restructure','rename','decouple','extract'}

def classify_commit_task(subject,message):
    text=f'{subject} {message}'.lower(); repair=any(w in text for w in REPAIR_WORDS); refactor=any(w in text for w in REFACTOR_WORDS)
    if repair and not refactor: return 'repair'
    if refactor and not repair: return 'refactor'
    return None


def convert_commitpack_row(raw):
    if str(raw.get('license','')).lower().strip() not in OPEN_SOURCE_LICENSES: return None
    old=clean_code(str(raw.get('old_contents') or '')); new=clean_code(str(raw.get('new_contents') or ''))
    if not old or not new or old==new: return None
    if not (MIN_CODE_CHARS<=len(old)<=MAX_CODE_CHARS and MIN_CODE_CHARS<=len(new)<=MAX_CODE_CHARS): return None
    old_ok,_=parse_and_compile(old); new_ok,_=parse_and_compile(new)
    if not old_ok or not new_ok: return None
    subject=str(raw.get('subject') or '').strip(); message=str(raw.get('message') or '').strip()
    task=classify_commit_task(subject,message)
    if task is None: return None
    instruction=subject or message or ('Fix this.' if task=='repair' else 'Refactor this.')
    file_path=str(raw.get('new_file') or raw.get('old_file') or '')
    project_context=f'Existing Python file: {file_path}' if file_path else ''
    domain,subdomain=infer_domain(new,instruction)
    source_ref=':'.join(x for x in [str(raw.get('repos') or ''),str(raw.get('commit') or ''),file_path] if x)
    try:
        return build_dataset_row(task=task,instruction=instruction,code_before=old,error='',project_context=project_context,
            target_code=new,domain=domain,subdomain=subdomain,difficulty=infer_difficulty(new),
            instruction_style=infer_instruction_style(instruction),concepts=infer_concepts(new,instruction),
            libraries=extract_third_party_libraries(new),verified=True,verification_level='structural',
            source_type='public_transformed',source_dataset='bigcode/commitpackft:python',source_reference=source_ref)
    except Exception: return None

print('CommitPackFT adapter ready.')

# Cell 23 - Stream CommitPackFT and show full converted examples

In [ ]:
stream=load_dataset('json',data_files=COMMITPACK_PY_URL,split='train',streaming=True)
preview=[]
for raw in stream:
    row=convert_commitpack_row(raw)
    if row is not None: preview.append(row)
    if len(preview)>=3: break
commitpack_preview_df=pd.DataFrame(preview,columns=DATASET_COLUMNS)
print(commitpack_preview_df.to_string(index=False))

# Cell 24 - Build CommitPackFT shard

In [ ]:
existing=len(read_jsonl(COMMITPACK_JSONL)); needed=max(COMMITPACK_ACCEPTED_TARGET-existing,0)
accepted_count=0; duplicate_count=0; scanned=0
print('Existing:',existing,'Need:',needed)
if needed:
    stream=load_dataset('json',data_files=COMMITPACK_PY_URL,split='train',streaming=True)
    for raw in tqdm(stream,desc='CommitPackFT conversion'):
        scanned+=1; row=convert_commitpack_row(raw)
        if row is None: continue
        accepted,reason=store.add(row,COMMITPACK_JSONL)
        if accepted: accepted_count+=1
        else: duplicate_count+=1
        if accepted_count>=needed: break
print('Scanned:',scanned,'Accepted:',accepted_count,'Dedup:',duplicate_count)
display(store.counts())

# Section C - New / Unfamiliar Library Generalization

This is a required training category, not a bonus.

The base model may already know common frameworks. That does not prove it can integrate an API it never saw in pretraining. The notebook therefore creates fictional packages with explicit signatures and contracts.

Qwen receives the API in `library_context`; the deterministic IR/compiler remains library-agnostic.

Mature target: roughly **10-20%** of the dataset, ideally around **15%**.

# Cell 25 - Synthetic library families

In [ ]:
SYLLABLES=['nova','zephyr','orbit','lumen','quanta','vertex','ember','aero','helios','cobalt','arc','flux']

def synthetic_package_name(kind,rng): return f'{rng.choice(SYLLABLES)}_{kind}_{rng.randint(1000,9999)}'

def make_synthetic_library_spec(index):
    rng=random.Random(SEED+100000+index); kind=rng.choice(['cache','http','db','queue','storage','auth','telemetry','flags']); package=synthetic_package_name(kind,rng)
    if kind=='cache':
        client=rng.choice(['CacheBridge','CacheClient','EntryCache']); getn=rng.choice(['read_entry','fetch_value','lookup']); setn=rng.choice(['write_entry','store_value','put'])
        context=f"""Package: {package}\n\nclass {client}:\n    def __init__(self, endpoint: str): ...\n    async def {getn}(self, key: str) -> bytes | None: ...\n    async def {setn}(self, key: str, value: bytes, ttl_seconds: int) -> None: ...\n    async def close(self) -> None: ...\n\nContract:\n- methods are asynchronous\n- ttl_seconds must be positive\n- close owned clients"""
        domain,subdomain='software_design','new_library_integration'; requirement='Create an async repository-like component that caches serialized user records for 60 seconds and falls back to a supplied async loader on misses.'; symbols=[client,getn,setn]
    elif kind=='http':
        client=rng.choice(['OrbitClient','WebBridge','AsyncTransport']); requestn=rng.choice(['send','perform','exchange'])
        context=f"""Package: {package}\n\nclass {client}:\n    async def {requestn}(self, method: str, url: str, *, json_body: dict | None = None, timeout_seconds: float = 10.0) -> Response: ...\n\nclass Response:\n    status_code: int\n    def json(self) -> dict: ...\n    def raise_for_status(self) -> None: ...\n\nContract:\n- {requestn} is asynchronous\n- Response.json() is synchronous"""
        domain,subdomain='networking','new_library_integration'; requirement='Build a typed async Orders API client. Retry 429 and 503 up to three attempts with exponential backoff, but do not retry other 4xx responses.'; symbols=[client,requestn,'raise_for_status']
    elif kind=='db':
        db=rng.choice(['DataHub','Database','Store']); tx=rng.choice(['Transaction','UnitSession','DbTransaction']); fetch=rng.choice(['fetch_one','select_one','get_row']); execute=rng.choice(['execute','run','apply'])
        context=f"""Package: {package}\n\nclass {db}:\n    def transaction(self) -> AsyncContextManager[{tx}]: ...\n\nclass {tx}:\n    async def {fetch}(self, query: str, params: dict) -> Row | None: ...\n    async def {execute}(self, query: str, params: dict) -> None: ...\n\nContract:\n- transaction() is used with async with\n- success commits and exceptions roll back\n- query values must use params, not interpolation"""
        domain,subdomain='database','new_library_integration'; requirement='Implement an atomic balance transfer between two accounts. Reject non-positive amounts and insufficient balance. Keep query values parameterized.'; symbols=[db,tx,fetch,execute]
    elif kind=='queue':
        cls=rng.choice(['MessageQueue','AsyncQueueClient','Broker']); pub=rng.choice(['publish','emit','send_message']); consume=rng.choice(['messages','consume','stream'])
        context=f"""Package: {package}\n\nclass {cls}:\n    async def {pub}(self, topic: str, payload: bytes) -> str: ...\n    async def {consume}(self, topic: str) -> AsyncIterator[Message]: ...\n    async def acknowledge(self, message_id: str) -> None: ...\n\nclass Message:\n    id: str\n    payload: bytes\n\nContract:\n- consumer iteration is asynchronous\n- acknowledge only after successful processing"""
        domain,subdomain='data_engineering','new_library_integration'; requirement='Create an async worker that processes messages with bounded concurrency, acknowledges only successful messages, and lets cancellation propagate.'; symbols=[cls,pub,consume,'acknowledge']
    elif kind=='storage':
        cls=rng.choice(['ObjectStore','BlobClient','BucketStore']); put=rng.choice(['put_object','upload_bytes','store_blob']); get=rng.choice(['get_object','download_bytes','read_blob'])
        context=f"""Package: {package}\n\nclass {cls}:\n    async def {put}(self, bucket: str, key: str, data: bytes) -> None: ...\n    async def {get}(self, bucket: str, key: str) -> bytes: ...\n    async def exists(self, bucket: str, key: str) -> bool: ...\n\nContract:\n- bucket and key are separate arguments\n- all operations are asynchronous"""
        domain,subdomain='data_engineering','new_library_integration'; requirement='Create an async artifact repository that stores JSON documents as UTF-8 bytes and retrieves and decodes them.'; symbols=[cls,put,get]
    elif kind=='auth':
        cls=rng.choice(['TokenVerifier','IdentityClient','AuthGateway']); verify=rng.choice(['verify_token','validate_access','inspect_token'])
        context=f"""Package: {package}\n\nclass {cls}:\n    def __init__(self, issuer: str, audience: str): ...\n    async def {verify}(self, token: str) -> Claims: ...\n\nclass Claims:\n    subject: str\n    scopes: frozenset[str]\n    expires_at: datetime\n\nclass InvalidToken(Exception): ...\n\nContract:\n- {verify} performs signature and expiry verification\n- do not decode/trust token payload manually"""
        domain,subdomain='security','new_library_integration'; requirement='Implement an async authorization helper that validates a bearer token and requires a specified scope before returning the authenticated subject.'; symbols=[cls,verify,'InvalidToken']
    elif kind=='telemetry':
        cls=rng.choice(['Telemetry','Tracer','ObservabilityClient']); span=rng.choice(['span','start_span','trace'])
        context=f"""Package: {package}\n\nclass {cls}:\n    def {span}(self, name: str) -> ContextManager[Span]: ...\n\nclass Span:\n    def set_attribute(self, key: str, value: str | int | float | bool) -> None: ...\n    def record_exception(self, exc: BaseException) -> None: ...\n\nContract:\n- {span} is a synchronous context manager\n- exceptions should be recorded and re-raised"""
        domain,subdomain='reliability','new_library_integration'; requirement='Wrap a payment operation with telemetry. Record payment_id as an attribute, record exceptions, and never swallow the original error.'; symbols=[cls,span,'record_exception']
    else:
        cls=rng.choice(['FlagClient','FeatureGate','ToggleClient']); method=rng.choice(['enabled','is_active','evaluate'])
        context=f"""Package: {package}\n\nclass {cls}:\n    async def {method}(self, flag_name: str, *, subject_id: str, default: bool = False) -> bool: ...\n\nContract:\n- evaluation is asynchronous\n- callers must supply subject_id\n- default is returned when the service cannot decide"""
        domain,subdomain='software_design','new_library_integration'; requirement='Create an async service method that selects a new checkout implementation when a feature flag is enabled for the current user, otherwise uses legacy checkout.'; symbols=[cls,method]
    return {'package':package,'library_context':textwrap.dedent(context).strip(),'domain':domain,'subdomain':subdomain,'suggested_requirement':requirement,'must_use_symbols':symbols}

print(json.dumps(make_synthetic_library_spec(0),indent=2))

# Cell 26 - Build Qwen requests for unfamiliar libraries

In [ ]:
def make_new_library_teacher_spec(index, task=None):
    lib=make_synthetic_library_spec(index); rng=random.Random(SEED+200000+index)
    task=task or rng.choice(TASKS)
    extra={
        'generate':f"Base the request on this goal: {lib['suggested_requirement']} Produce a focused reusable implementation.",
        'full_file_generate':f"Base the request on this goal: {lib['suggested_requirement']} Produce a complete importable module using the supplied SDK.",
        'repair':f"Create code that tries to satisfy this goal but misuses the supplied SDK or has a meaningful semantic bug: {lib['suggested_requirement']}",
        'refactor':f"Create working code using the supplied SDK, then refactor it while preserving behavior: {lib['suggested_requirement']}",
    }[task]
    return {
        'task':task,'domain':lib['domain'],'subdomain':lib['subdomain'],
        'difficulty':rng.choice(['intermediate','advanced','advanced','expert']),
        'instruction_style':rng.choice(STYLES),
        'project_context':f"The package `{lib['package']}` is available. Treat supplied API context as authoritative.",
        'library_context':lib['library_context'],'library_package':lib['package'],'must_use_symbols':lib['must_use_symbols'],
        'extra_requirements':extra,'source_reference':f"newlib:{lib['package']}:{task}:{index}"}

example_newlib_spec=make_new_library_teacher_spec(0,'generate')
print(build_teacher_prompt(example_newlib_spec))

# Cell 27 - Real unfamiliar-library demo

This is the key generalization test: Qwen gets a fictional API and must integrate that exact package rather than relying on memorized popular libraries.

In [ ]:
spec=make_new_library_teacher_spec(123,'generate')
row,errors=generate_teacher_row(spec,source_type='new_library_teacher')
if row is None:
    print('FAILED:',errors)
else:
    for column in DATASET_COLUMNS:
        print(f'\n### {column}\n{row[column]}')
    print('\nFULL 24-COLUMN ROW')
    print(pd.DataFrame([row],columns=DATASET_COLUMNS).to_string(index=False))

# Cell 28 - Build unfamiliar-library shard

In [ ]:
existing=len(read_jsonl(NEW_LIBRARY_JSONL)); needed=max(NEW_LIBRARY_ACCEPTED_TARGET-existing,0)
accepted_count=0; rejected_count=0; index=existing
bar=tqdm(total=needed,desc='New-library Qwen')
while accepted_count<needed:
    spec=make_new_library_teacher_spec(index); index+=1
    row,errors=generate_teacher_row(spec,source_type='new_library_teacher')
    if row is None:
        rejected_count+=1; append_jsonl(REJECTED_JSONL,{'pipeline':'new_library_teacher','spec':spec,'errors':errors,'timestamp':time.time()}); continue
    accepted,reason=store.add(row,NEW_LIBRARY_JSONL)
    if accepted:
        accepted_count+=1; bar.update(1)
    else: rejected_count+=1
    if accepted_count and accepted_count%10==0: gc.collect(); torch.cuda.empty_cache()
bar.close()
print('Accepted:',accepted_count,'Rejected:',rejected_count)
display(store.counts())

# Section D - Optional Qwen enrichment of public rows

Public commit messages are useful provenance but are not always realistic user prompts. This optional helper rewrites only the instruction while preserving the public target code. It is disabled by default because it consumes teacher GPU time.

In [ ]:
ENRICH_PUBLIC_INSTRUCTIONS=False

def rewrite_public_instruction_with_qwen(row):
    prompt=f"""Rewrite the source instruction into a natural realistic developer request.\nPreserve exact intended behavior. Do not add unsupported requirements.\nReturn JSON only: {{"instruction":"..."}}\n\nTask: {row['task']}\nOriginal: {row['instruction']}\nExisting code:\n{row['code_before'][:6000]}\nTarget code:\n{row['target_code'][:6000]}"""
    obj=qwen_generate_json(prompt,temperature=0.3); value=str(obj.get('instruction') or '').strip()
    if not value: raise ValueError('Empty rewritten instruction')
    return value

print('Optional enrichment ready. Enabled:',ENRICH_PUBLIC_INSTRUCTIONS)

# Section E - Optional execution-based verification

Structural parse/compile checks are always performed. Executing model-generated Python is stronger but is not a hardened sandbox, so it is disabled by default.

For production, run tests inside an isolated container/sandbox.

In [ ]:
def run_pytest(source_code,tests_code,timeout_seconds=PYTEST_TIMEOUT_SECONDS):
    if not tests_code.strip(): return False,'no tests'
    with tempfile.TemporaryDirectory() as tmp:
        p=Path(tmp); (p/'solution.py').write_text(source_code,encoding='utf-8'); (p/'test_solution.py').write_text(tests_code,encoding='utf-8')
        env=os.environ.copy(); env['PYTHONHASHSEED']='0'
        try:
            result=subprocess.run([sys.executable,'-m','pytest','-q','test_solution.py'],cwd=tmp,env=env,capture_output=True,text=True,timeout=timeout_seconds)
        except subprocess.TimeoutExpired:
            return False,'pytest timeout'
        return result.returncode==0,(result.stdout+'\n'+result.stderr).strip()


def upgrade_row_verification(row):
    if not RUN_GENERATED_TESTS or not row.get('tests','').strip(): return row
    ok,out=run_pytest(row['target_code'],row['tests'])
    if not ok: raise ValueError('Target tests failed:\n'+out[-3000:])
    if row['task']=='repair':
        before_ok,_=run_pytest(row['code_before'],row['tests'])
        if before_ok: raise ValueError('Repair before unexpectedly passes tests')
    elif row['task']=='refactor':
        before_ok,before_out=run_pytest(row['code_before'],row['tests'])
        if not before_ok: raise ValueError('Refactor before fails tests:\n'+before_out[-3000:])
    row=dict(row); row['verified']=True; row['verification_level']='tests'; return row

print('Execution verification enabled:',RUN_GENERATED_TESTS)

# Section F - Merge, audit, balance, and export

All shards now share exactly the same schema.

# Cell 30 - Load and merge all shards

In [ ]:
all_rows=[]
for path in SOURCE_FILES:
    rows=read_jsonl(path); print(path.name,'->',len(rows)); all_rows.extend(rows)
print('Raw merged:',len(all_rows))
dedup={}
for row in all_rows: dedup[row['content_hash']]=row
all_rows=list(dedup.values())
print('After exact hash dedup:',len(all_rows))
all_df=pd.DataFrame(all_rows,columns=DATASET_COLUMNS)
display(all_df.head(5))

# Cell 31 - Quality audit

In [ ]:
def audit_dataframe(df):
    if df.empty: return {'rows':0}
    assert list(df.columns)==DATASET_COLUMNS
    assert df['target_code'].astype(str).str.strip().ne('').all()
    assert df['target_ir'].astype(str).str.strip().ne('').all()
    assert df['task'].isin(TASKS).all()
    assert df['target_mode'].isin(['build','edit']).all()
    edit=df[df['target_mode']=='edit']
    if len(edit): assert edit['code_before'].astype(str).str.strip().ne('').all()
    return {
        'rows':int(len(df)),'columns':int(len(df.columns)),
        'unique_ids':bool(df['id'].nunique()==len(df)),
        'unique_hashes':bool(df['content_hash'].nunique()==len(df)),
        'new_library_fraction':float((df['source_type']=='new_library_teacher').mean()),
        'task_distribution':df['task'].value_counts().to_dict(),
        'source_distribution':df['source_type'].value_counts().to_dict(),
        'difficulty_distribution':df['difficulty'].value_counts().to_dict(),
        'style_distribution':df['instruction_style'].value_counts().to_dict(),
        'verification_distribution':df['verification_level'].value_counts().to_dict(),
        'domain_distribution':df['domain'].value_counts().to_dict(),
    }
audit=audit_dataframe(all_df)
print(json.dumps(audit,indent=2,ensure_ascii=False))

# Cell 32 - Optional mature 50K-per-task balance

Early runs export all accepted rows. When the candidate pool is large enough, set `BUILD_BALANCED_FINAL=True` to sample exactly 50K rows per task.

In [ ]:
BUILD_BALANCED_FINAL=False

def task_deficits(df,targets):
    counts=df['task'].value_counts().to_dict() if len(df) else {}
    return {task:max(target-int(counts.get(task,0)),0) for task,target in targets.items()}

def build_balanced_dataset(df,targets,random_state=SEED):
    parts=[]
    for task,target in targets.items():
        subset=df[df['task']==task]
        if len(subset)<target: raise ValueError(f'{task} has {len(subset):,}, needs {target:,}')
        parts.append(subset.sample(target,random_state=random_state))
    return pd.concat(parts,ignore_index=True).sample(frac=1,random_state=random_state).reset_index(drop=True)

print('Current deficits:')
print(json.dumps(task_deficits(all_df,FINAL_TASK_TARGETS),indent=2))
final_df=build_balanced_dataset(all_df,FINAL_TASK_TARGETS) if BUILD_BALANCED_FINAL else all_df.sample(frac=1,random_state=SEED).reset_index(drop=True)
print('Final selected rows:',len(final_df))

# Cell 33 - New-library coverage audit

In [ ]:
if len(final_df):
    mask=final_df['source_type'].eq('new_library_teacher'); fraction=float(mask.mean())
    print(f'New-library rows: {int(mask.sum()):,}')
    print(f'New-library fraction: {fraction:.2%}')
    print(f'Preferred mature range: {NEW_LIBRARY_MIN_FRACTION:.0%}-{NEW_LIBRARY_MAX_FRACTION:.0%}')
    if BUILD_BALANCED_FINAL and fraction<NEW_LIBRARY_MIN_FRACTION:
        print('WARNING: generate more unfamiliar-library rows before freezing the dataset.')

# Cell 34 - Show complete final rows and all 24 columns

This intentionally prints long code, IR, tests, and library context in full.

In [ ]:
def show_complete_rows(df,n=4,random_state=SEED):
    if df.empty:
        print('Dataset is empty.'); return
    sample=df.sample(min(n,len(df)),random_state=random_state)
    print(sample[DATASET_COLUMNS].to_string(index=False))
    for row_number,(_,row) in enumerate(sample.iterrows(),1):
        print('\n'+'='*120); print(f'COMPLETE ROW {row_number}'); print('='*120)
        for column in DATASET_COLUMNS:
            print(f'\n[{column}]\n{row[column]}')
show_complete_rows(final_df,min(4,len(final_df)))

# Cell 35 - Export JSONL, CSV, Parquet, and summary

JSONL is the recommended canonical format because source, tests, library context, and IR are multiline. CSV is included for inspection.

In [ ]:
with FINAL_JSONL.open('w',encoding='utf-8') as f:
    for row in final_df.to_dict(orient='records'): f.write(json.dumps(row,ensure_ascii=False)+'\n')
final_df.to_csv(FINAL_CSV,index=False,encoding='utf-8',quoting=csv.QUOTE_ALL)
final_df.to_parquet(FINAL_PARQUET,index=False)
summary=audit_dataframe(final_df)
summary.update({'student_model':STUDENT_MODEL_ID,'teacher_model':TEACHER_MODEL_ID,'ir_version':IR_VERSION,'dataset_columns':DATASET_COLUMNS,'task_targets':FINAL_TASK_TARGETS})
FINAL_SUMMARY.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
jsonl_rt=read_jsonl(FINAL_JSONL); csv_rt=pd.read_csv(FINAL_CSV); parquet_rt=pd.read_parquet(FINAL_PARQUET)
assert len(jsonl_rt)==len(final_df)==len(csv_rt)==len(parquet_rt)
assert list(csv_rt.columns)==DATASET_COLUMNS and list(parquet_rt.columns)==DATASET_COLUMNS
print('JSONL:',FINAL_JSONL); print('CSV:',FINAL_CSV); print('Parquet:',FINAL_PARQUET); print('Summary:',FINAL_SUMMARY)
print(json.dumps(summary,indent=2,ensure_ascii=False))

# Cell 36 - Preview the future CodeT5+ training pair

The student should only receive information available at inference time. Metadata such as difficulty, source, hashes, and verification status is for dataset management, not normal model input.

In [ ]:
def serialize_student_input(row):
    sections=[('TASK',row.get('task','')),('INSTRUCTION',row.get('instruction','')),('PROJECT_CONTEXT',row.get('project_context','')),('LIBRARY_CONTEXT',row.get('library_context','')),('CODE_BEFORE',row.get('code_before','')),('ERROR',row.get('error',''))]
    parts=[]
    for tag,value in sections:
        value=str(value or '').strip()
        if value: parts.append(f'<{tag}>\n{value}\n</{tag}>')
    return '\n\n'.join(parts)
if len(final_df):
    example=final_df.iloc[0].to_dict()
    print('='*120+'\nCODET5+ INPUT\n'+'='*120); print(serialize_student_input(example))
    print('\n'+'='*120+'\nCODET5+ TARGET IR\n'+'='*120); print(example['target_ir'])

# Cell 37 - Proposed IR special tokens for CodeT5+

Only stable grammar markers become special tokens. Package names, library methods, identifiers, types, strings, and literals remain ordinary tokenizer text.

In [ ]:
IR_SPECIAL_TOKENS=[
 '<BUILD>','<EDIT>','<END>','<MODULE>','<END_MODULE>','<IMPORT>','<END_IMPORT>','<IMPORT_FROM>','<END_IMPORT_FROM>','<MODULE_NAME>',
 '<FUNCTION>','<END_FUNCTION>','<ASYNC_FUNCTION>','<END_ASYNC_FUNCTION>','<CLASS>','<END_CLASS>','<PARAM>','<END_PARAM>','<PARAM_KIND>',
 '<NAME>','<NAME_REF>','<TYPE>','<RETURN_TYPE>','<BODY>','<END_BODY>','<DOCSTRING>','<DECORATOR>','<END_DECORATOR>',
 '<ASSIGN>','<END_ASSIGN>','<ANN_ASSIGN>','<END_ANN_ASSIGN>','<AUG_ASSIGN>','<END_AUG_ASSIGN>','<TARGET>','<END_TARGET>','<VALUE>','<END_VALUE>',
 '<RETURN>','<END_RETURN>','<EXPR_STMT>','<END_EXPR_STMT>','<IF>','<END_IF>','<ELSE>','<END_ELSE>','<CONDITION>','<END_CONDITION>',
 '<FOR>','<END_FOR>','<ASYNC_FOR>','<END_ASYNC_FOR>','<ITER>','<END_ITER>','<WHILE>','<END_WHILE>','<WITH>','<END_WITH>','<ASYNC_WITH>','<END_ASYNC_WITH>',
 '<WITH_ITEM>','<END_WITH_ITEM>','<AS>','<END_AS>','<TRY>','<END_TRY>','<EXCEPT>','<END_EXCEPT>','<FINALLY>','<END_FINALLY>','<RAISE>','<END_RAISE>',
 '<ASSERT>','<END_ASSERT>','<CALL>','<END_CALL>','<FUNC>','<END_FUNC>','<ARG>','<END_ARG>','<KWARG>','<END_KWARG>','<KW_NAME>',
 '<ATTRIBUTE>','<END_ATTRIBUTE>','<BASE>','<END_BASE>','<ATTR_NAME>','<AWAIT>','<END_AWAIT>','<BINOP>','<END_BINOP>','<OP>','<LEFT>','<END_LEFT>','<RIGHT>','<END_RIGHT>',
 '<BOOL_OP>','<END_BOOL_OP>','<UNARY_OP>','<END_UNARY_OP>','<COMPARE>','<END_COMPARE>','<COMPARE_OP>','<END_COMPARE_OP>','<SUBSCRIPT>','<END_SUBSCRIPT>','<SLICE>','<END_SLICE>',
 '<CONST>','<LIST>','<END_LIST>','<TUPLE>','<END_TUPLE>','<SET>','<END_SET>','<DICT>','<END_DICT>','<ITEM>','<END_ITEM>','<KEY>','<END_KEY>','<DICT_UNPACK>',
 '<PY_EXPR>','<PY_STMT>','<PASS>','<BREAK>','<CONTINUE>','<DEFAULT>','<END_DEFAULT>','<EXCEPTION_TYPE>','<END_EXCEPTION_TYPE>'
]
IR_SPECIAL_TOKENS=list(dict.fromkeys(IR_SPECIAL_TOKENS))
print('Special token count:',len(IR_SPECIAL_TOKENS)); print(IR_SPECIAL_TOKENS)

# Cell 38 - Save reproducible generation manifest

In [ ]:
manifest={
 'student_model':STUDENT_MODEL_ID,'teacher_model':TEACHER_MODEL_ID,'seed':SEED,'demo_mode':DEMO_MODE,'ir_version':IR_VERSION,
 'generation_max_total_tokens':GENERATION_MAX_TOTAL_TOKENS,'run_generated_tests':RUN_GENERATED_TESTS,
 'strict_structure_dedup':STRICT_STRUCTURE_DEDUP,'final_task_targets':FINAL_TASK_TARGETS,
 'new_library_target_range':[NEW_LIBRARY_MIN_FRACTION,NEW_LIBRARY_IDEAL_FRACTION,NEW_LIBRARY_MAX_FRACTION],
 'public_sources':{'magicoder':MAGICODER_DATASET_ID,'commitpack_python_url':COMMITPACK_PY_URL},
 'schema':DATASET_COLUMNS,'ir_special_tokens':IR_SPECIAL_TOKENS}
manifest_path=ROOT_DIR/'generation_manifest.json'; manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(manifest_path)

# Cell 39 - Download final artifacts from Colab

In [ ]:
from google.colab import files
for path in [FINAL_JSONL,FINAL_CSV,FINAL_PARQUET,FINAL_SUMMARY,ROOT_DIR/'generation_manifest.json']:
    if path.exists():
        print('Downloading:',path.name); files.download(str(path))

# Cell 40 - Scale plan and next validation gates

Recommended progression:

```text
Stage 1: tiny demo - inspect complete rows manually
Stage 2: ~400 accepted rows - 100 per task
Stage 3: ~2,000 rows - inspect rejection and duplication patterns
Stage 4: ~10,000 rows - first CodeT5+ 220M fine-tuning
Stage 5: ~50,000 rows - evaluate whether IR helps versus direct Python
Stage 6: build candidate pool larger than 200K
Stage 7: balance/freeze 50K per task
```

For production scale:

1. Use Google Drive checkpointing.
2. Transform public data first because it is cheap.
3. Generate unfamiliar-library rows until the target share is available.
4. Compute task/domain/difficulty deficits.
5. Use Qwen to fill missing coverage cells.
6. Add hardened execution-based verification.
7. Add MinHash/semantic near-duplicate filtering.
8. Reserve completely unseen synthetic library families for evaluation.

## Important IR v1 choice

Repair/refactor currently learn:

```text
old code in encoder -> complete corrected structured target program
```

rather than fine-grained edit commands. This is deliberate. The dataset preserves `code_before` and `target_code`, so a future tree-diff IR can be regenerated and compared as an ablation.